Run by Member 4. Skipped 3E, 4B, 4C, Step 6. Final numbers match: 5,665 / 100,740 / 51,318.”

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir("/content/drive/MyDrive/esci/dataset_v2.3")   # output files are saved here
print(os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/esci/dataset_v2.3


# 02 · ESCI electronics dataset — v2.3 (decision-aligned rebuild)

**Run order:** Step 1A → 1B (done) → Step 2 → Step 3 → Step 4 → label → 4C. See the stop cell at the bottom.
Nothing in this notebook writes or overwrites a dataset yet.

## Decision log (what this notebook implements)

| # | Decision | Status |
|---|---|---|
| D1 | US locale only (ESCI also has `es`, `jp`; our DistilBERT / SBERT pipeline is English) | carried over |
| D2 | ESCI has **no category field** → electronics is inferred from text (keywords + vetoes) | carried over, now stated explicitly |
| D3 | Match on **brand + title** with word boundaries; bullets/description are *not* used (they list compatible devices) | carried over |
| D4 | Keyword list = the 5 original inclusion groups **+ 3 groups added in v2.2**: `computer_peripherals` (monitors, keyboards, mice, printers), `gaming` (game controllers), `smart_home` (smart plugs/bulbs/switches); microphones → `audio`, camera lenses → `imaging`. RAM context-aware, routers need network context, laptop product lines added | **new / changed** |
| D5 | Brand alone never qualifies. Brand + model token qualifies; Samsung/Sony/LG/Panasonic/Philips/Sharp/Toshiba additionally need a hardware noun | carried over + tightened |
| D6 | Vetoes: accessory head nouns (query-only strict forms), homonyms, outdoor/power tools, woodworking routers, furniture, exercise gear, fashion/novelty, phone plans/SIM kits, optics, books, case brands, replacement parts, consumables (ink, paper, filament), new-scope guards (baby/medical monitors, piano keyboards, mouse pests/pads, contact lenses). **Remaining out-of-scope buckets**: game consoles (game titles carry console names), projectors, drones, e-readers, scanners — each re-admittable via `allow_buckets` | carried over + extended |
| D7 | **Query-level decision** from the query's Exact/Substitute products; **keep the query's full judged list** (ESCI's original design: ~20 judged results per query) | **changed** (old filter kept only electronics-matching product rows) |
| D8 | Preserve ESCI metadata: `split`, `small_version`, `large_version`, `source` | **new** |
| D9 | NDCG gains = ESCI paper values **E=1.0, S=0.1, C=0.01, I=0.0** | **new** (proposal currently says 1.0/0.5/0.1/0.0) |
| D10 | Classifier pool: sources `other` + `parse_pattern`, must have ≥1 E/S product, length buckets 90/230/110/70, off-topic queries labelled `N/A` | carried over |
| D11 | Classifier queries drawn from ESCI **train** split only; test split untouched until the final ablation | **new** |
| D13 | **v2.3 is the final patch and is frozen** (version-tagged, thresholds a = 0.50, b = 0.80, n_min = 3). Nothing is patched after the audit; residual errors are *reported*, not fixed | **new** |
| D12 | Re-audit at **query level** on a **fresh, blind, stratified** sample (seed 2026, ~310 queries, excluding every query seen during development): stratified estimators for precision *and* recall with bootstrap 95 % CIs | **new** (old audit = 200 products, precision only, same rows used for patching) |

In [ ]:
import re
import pandas as pd
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

DATA = "/content/drive/MyDrive/esci/data/"
sources  = pd.read_csv(DATA + "shopping_queries_dataset_sources.csv")
products = pd.read_parquet(DATA + "shopping_queries_dataset_products.parquet")
examples = pd.read_parquet(DATA + "shopping_queries_dataset_examples.parquet")

print(examples.columns.tolist())
us_products = products[products["product_locale"] == "us"].copy()
us_examples = examples[examples["product_locale"] == "us"].copy()
print(len(us_products), "US products |", len(us_examples), "US judgment rows |", us_examples["query_id"].nunique(), "US queries")

['example_id', 'query', 'query_id', 'product_id', 'product_locale', 'esci_label', 'small_version', 'large_version', 'split']
1215854 US products | 1818825 US judgment rows | 97345 US queries


## Step 1A — Raw ESCI structure (US)
Reference points from the ESCI paper (Reddy et al., 2022): US large version = 97,345 queries / 1,819,105 judgments (≈18.7 per query);
US small version = 29,844 queries / 601,462 judgments (≈20.2 per query); official split ≈ 70/15/15.

In [ ]:
q_raw = us_examples.groupby("query_id").agg(
    n_judged=("example_id", "size"),
    split=("split", "first"),
    small=("small_version", "max"),
    large=("large_version", "max"),
)

print("US queries:", len(q_raw))
print("\nqueries by split:\n", q_raw["split"].value_counts())
print("\nsmall_version flag x split (queries):\n", pd.crosstab(q_raw["split"], q_raw["small"]))
print("\njudged products per query (US, all):\n", q_raw["n_judged"].describe().round(2))
print("\nlabel mix (US, all):\n", us_examples["esci_label"].value_counts(normalize=True).round(4))
print("\nquery sources (US queries):\n",
      q_raw.join(sources.set_index("query_id"))["source"].value_counts())

US queries: 97345

queries by split:
 split
train    74887
test     22458
Name: count, dtype: int64

small_version flag x split (queries):
 small      0      1
split              
test   13502   8956
train  53999  20888

judged products per query (US, all):
 count    97345.00
mean        18.68
std          7.98
min          8.00
25%         16.00
50%         16.00
75%         16.00
max        198.00
Name: n_judged, dtype: float64

label mix (US, all):
 esci_label
E    0.6859
S    0.2031
I    0.0890
C    0.0220
Name: proportion, dtype: float64

query sources (US queries):
 source
other            88774
parse_pattern     4250
negations         2899
behavioral        1253
nlqec              169
Name: count, dtype: int64


## Step 1B — What did the *current* filter do to ESCI's original list structure?
`old_filter` is an exact copy of the filter in `01_ESCi_Dataset_Exploration.ipynb` (final state, incl. the outdoor-tool exclusion).
Expected reproduction: **60,664 products → 17,919 queries → 122,004 rows**. If you see different numbers, tell me before anything else.

In [ ]:
OLD_KEYWORDS = [
    "phone", "smartphone", "laptop", "notebook computer", "tablet", "desktop computer", "smartwatch", "chromebook",
    "ram", "ddr4", "ddr5", "ssd", "hard drive", "processor", "cpu", "gpu", "motherboard",
    "power bank", "charger", "usb cable", "hdmi cable",
    "camera", "dash cam", "webcam", "security camera", "camcorder",
    "television", " tv ", "smart speaker", "router", "wifi extender", "bluetooth speaker",
    "headphone", "earbud", "earphone",
]
OLD_BRANDS = ["samsung", "apple", "xiaomi", "sony", "lg", "dell", "hp", "lenovo", "asus", "acer", "panasonic", "canon",
              "nikon", "gopro", "logitech", "jbl", "bose", "anker", "tp-link", "netgear", "corsair", "kingston",
              "western digital", "seagate"]
OLD_HW = (r"\b(?:player|console|headphones?|speakers?|lens|camera|batter(?:y|ies)|receiver|soundbar|walkman|radio|monitor"
          r"|vcr|changer|memory card|earbuds?|earphones?|turntable|amplifier)\b")
OLD_EXCL = (r"\b(?:food processor|dodge ram|ram \d{3,4}|ram trucks?|ram pickup|ram hitch|dodge charger|charger challenger"
            r"|wood(?:en)? router|router table|as seen on tv|tv show|writing tablet|the emerald tablet)\b")
OLD_BOOK = r"\b(?:a novel|: a novel|a memoir)\b"
OLD_TOOLS = (r"\b(?:edger|trimmer|chainsaw|lawn\s?mower|leaf\s?blower|hedge\s?trimmer|pole\s?saw|weed\s?wacker)\b")

def old_filter(us_products):
    title = us_products["product_title"].str.lower().fillna("")
    brand = us_products["product_brand"].str.lower().fillna("")
    pattern = r"\b(?:" + "|".join(re.escape(k) for k in OLD_KEYWORDS) + r")\b"
    keyword_mask = title.str.contains(pattern, regex=True)
    brand_mask = brand.isin(OLD_BRANDS)
    is_cong = brand.isin(["samsung", "sony", "lg", "panasonic"])
    hw_mask = title.str.contains(OLD_HW, regex=True)
    brand_final = brand_mask & ~(is_cong & ~keyword_mask & ~hw_mask)
    excl = title.str.contains(OLD_EXCL, regex=True)
    book = (brand_mask & ~keyword_mask) & title.str.contains(OLD_BOOK, regex=True)
    out = us_products[(keyword_mask | brand_final) & ~excl & ~book]
    tools = title.loc[out.index].str.contains(OLD_TOOLS, regex=True)
    return out[~tools]

old = old_filter(us_products)
old_ids = set(old["product_id"])
old_ex = us_examples[us_examples["product_id"].isin(old_ids)]
print(len(old), "products   (expect 60,664)")
print(old_ex["query_id"].nunique(), "queries    (expect 17,919)")
print(len(old_ex), "rows       (expect 122,004)")

# ---- depth collapse: retained rows vs the query's full ESCI list
kept_n = old_ex.groupby("query_id").size().rename("n_retained")
full_n = us_examples[us_examples["query_id"].isin(kept_n.index)].groupby("query_id").size().rename("n_full_list")
cmp = pd.concat([kept_n, full_n], axis=1)
cmp["share_retained"] = cmp["n_retained"] / cmp["n_full_list"]
print("\njudged products per query: retained vs full ESCI list")
print(cmp.describe().round(2))
print("\nqueries with < 10 retained products (NDCG@10 not meaningful):", round((cmp["n_retained"] < 10).mean(), 3))
print("rows removed from these queries' lists:", int(cmp["n_full_list"].sum() - cmp["n_retained"].sum()))

# ---- label mix: full list vs retained rows
full_rows = us_examples[us_examples["query_id"].isin(kept_n.index)]
print("\nlabel mix, full list vs retained rows")
print(pd.DataFrame({"full_list": full_rows["esci_label"].value_counts(normalize=True),
                    "retained": old_ex["esci_label"].value_counts(normalize=True)}).round(3))
print("\nqueries with no Exact product among retained rows:",
      round(1 - old_ex.groupby("query_id")["esci_label"].apply(lambda s: (s == "E").any()).mean(), 3))

# ---- where do the 17,919 queries sit in ESCI's own splits / versions?
meta = q_raw.loc[kept_n.index]
print("\nsplit:\n", meta["split"].value_counts())
print("\nsmall_version flag:\n", meta["small"].value_counts())

60657 products   (expect 60,664)
17917 queries    (expect 17,919)
121996 rows       (expect 122,004)

judged products per query: retained vs full ESCI list
       n_retained  n_full_list  share_retained
count    17917.00     17917.00        17917.00
mean         6.81        18.95            0.38
std          6.92         8.84            0.34
min          1.00         8.00            0.01
25%          1.00        16.00            0.06
50%          4.00        16.00            0.25
75%         11.00        16.00            0.69
max        163.00       198.00            1.00

queries with < 10 retained products (NDCG@10 not meaningful): 0.698
rows removed from these queries' lists: 217592

label mix, full list vs retained rows
            full_list  retained
esci_label                     
E               0.666     0.688
S               0.192     0.210
I               0.106     0.070
C               0.035     0.032

queries with no Exact product among retained rows: 0.197

split:
 split
t

### Recorded results — Step 1 (for the logbook)
* ESCI US matches the paper: **97,345 queries**; small version **29,844** (train 20,888 / test 8,956); split train 74,887 / test 22,458.
* Judged products per US query: mean 18.68, median 16, max 198. Label mix (US): E 68.6 % · S 20.3 % · C 2.2 % · I 8.9 %.
* Old filter reproduced exactly: 60,664 products → 17,919 queries → 122,004 rows.
* Old filter shrank those queries' lists from a mean of **18.95 to 6.81** judged products (median 16 → 4; 25th percentile = 1). **69.8 %** of the queries have < 10 retained products; **19.7 %** have no Exact product left; 217,616 rows were removed.
* Irrelevant share fell 10.6 % → 7.0 % (hard negatives removed).
* 25 % of old-kept queries were admitted on a *single* electronics product → the old "any product matches" rule lets stray matches pull in off-topic queries.

## Step 2 — v2 filter definitions + unit tests (no dataset needed to run)
Merges the new keyword list with every decision from the earlier notebook and the 200-row audit.
The tests below are the audit failures plus true-electronics controls; **expect `failures: 0`**.

In [ ]:
import re
import pandas as pd

def _rx(parts):
    return re.compile(r"\b(?:" + "|".join(parts) + r")\b", re.I)

# ---------------- 1. CATEGORY TERMS ----------------
CATEGORY_TERMS = {
    "mobile_computing": [
        r"smart ?phones?", r"cell ?phones?", r"mobile phones?", r"iphones?", r"phones?",
        r"galaxy (?:s|a|z|m|note)\s?\d+\w*", r"pixel \d\w*", r"redmi", r"poco [a-z]\d+", r"oneplus", r"moto [a-z]\d*",
        r"laptops?", r"notebook (?:computers?|pcs?)",
        r"ideapad", r"thinkpad", r"thinkbook", r"vivobook", r"zenbook", r"inspiron", r"latitude", r"alienware",
        r"elitebook", r"probook", r"surface (?:pro|go|laptop)", r"galaxy book", r"chromebooks?", r"macbooks?", r"ultrabooks?",
        r"ipads?", r"galaxy tab\w*", r"fire (?:hd |\d{1,2} )?(?:kids )?tablets?",
        r"(?:android|windows|kids?|drawing|graphics?|pen|digital|hd|wi-?fi|lte|samsung|lenovo|amazon|huawei|apple|tcl|\d{1,2}(?:\.\d)?[- ]?(?:inch|in)) tablets?",
        r"tablets? (?:pc|computer|\d{1,2}(?:\.\d)? ?(?:inch|in)|for (?:kids|drawing|reading|gaming|seniors|toddlers|students|work))",
        r"(?:desktop|gaming|mini|all[- ]in[- ]one) (?:computers?|pcs?)",
        r"smart ?watch(?:es)?", r"apple watch", r"galaxy watch", r"fitbit", r"amazfit",
        r"fitness (?:trackers?|bands?)", r"garmin (?:forerunner|fenix|venu|vivoactive)",
    ],
    "components_accessories": [
        # RAM needs memory context (bare "ram" = Dodge Ram, RAM Golf, "ram" in other words)
        r"\d+\s?(?:gb|tb|mb)\s*(?:\w+\s+){0,2}ram", r"ram\s*\d+\s?(?:gb|tb|mb)",
        r"(?:laptop|desktop|computer|memory|pc|gaming|server) ram", r"ram (?:memory|kit|modules?|upgrade)s?",
        r"ddr[2-5]", r"so-?dimm", r"dimm",
        r"ssds?", r"solid[- ]state drives?", r"nvme", r"m\.2", r"hdds?", r"hard (?:disk )?drives?", r"external (?:hard )?drives?",
        r"(?:usb |flash |thumb |pen )drives?", r"micro ?sd(?:hc|xc)?(?: cards?)?", r"sd cards?",
        r"memory cards?", r"card readers?",
        r"cpus?", r"processors?", r"ryzen", r"intel core", r"core i[3579]", r"threadripper", r"xeon",
        r"gpus?", r"graphics cards?", r"video cards?", r"geforce", r"rtx ?\d{3,4}", r"gtx ?\d{3,4}",
        r"radeon", r"motherboards?", r"psu", r"(?:pc|computer|cpu|case|radiator) fans?",
        r"power ?banks?", r"portable chargers?", r"chargers?", r"charging (?:cables?|docks?|stations?)",
        r"usb[- ]?c", r"usb (?:type[- ]c |micro |mini )?cables?", r"micro usb", r"hdmi",
        r"displayport", r"lightning (?:cables?|chargers?)", r"ethernet cables?", r"cat ?[5-8]e? cables?",
        r"aux cables?", r"airtags?", r"usb hubs?", r"docking stations?", r"usb (?:adapters?|extension)",
    ],
    "imaging": [
        r"cameras?", r"dash ?cams?", r"web ?cams?", r"camcorders?", r"gopro", r"dslr", r"mirrorless",
        r"action cam(?:era)?s?", r"cctv", r"ip cameras?", r"video doorbells?", r"doorbell cam(?:era)?s?", r"ring doorbell", r"smart doorbell", r"nanny cams?",
        r"trail cameras?", r"instax", r"lens(?:es)?",                          # NEW scope: camera lenses
    ],
    "home_electronics": [
        r"tvs?", r"televisions?", r"fire tv(?: stick| cube)?", r"roku", r"chromecast",
        r"streaming (?:devices?|sticks?)", r"tv box", r"soundbars?", r"echo (?:dot|show|pop|studio)",
        r"homepod", r"google nest (?:mini|audio|hub|wi-?fi)",
        r"(?:bluetooth|smart|portable|wireless|computer|pc|party|waterproof|bookshelf|usb|home theater) speakers?",
        r"speakers?",
        # routers only with network context (bare "router" = woodworking tool)
        r"(?:wi-?fi|wireless|mesh|gaming|dual[- ]band|tri[- ]band|[345]g|lte|modem|gigabit|ethernet|vpn|smart) routers?",
        r"routers? (?:ac|ax)\d{3,4}", r"(?:wi-?fi|range|signal) extenders?", r"mesh wi-?fi", r"wi-?fi (?:system|6|7)",
        r"modems?", r"access points?", r"network switch(?:es)?", r"powerline adapters?",
        r"(?:desk|table|tower|pedestal|standing|stand|floor|usb|clip[- ]on|oscillating|rechargeable|"
        r"portable|box|window|bladeless|misting|personal|neck|battery[- ](?:operated|powered)) fans?",
        r"air circulators?",
    ],
    "audio": [
        r"head ?phones?", r"ear ?buds?", r"ear ?phones?", r"airpods?", r"headsets?", r"in-?ear",
        r"over-?ear", r"on-?ear", r"true wireless", r"tws", r"neckbands?", r"bone conduction",
        r"microphones?", r"mics?", r"lavalier",                                  # NEW scope: microphones
    ],
    # ---- NEW in scope (v2.2): monitors, keyboards, mice, printers ----
    "computer_peripherals": [
        r"monitors?", r"keyboards?", r"mice", r"mouse (?:for|with) (?:laptop|pc|computer|mac|ipad)",
        r"(?:computer|gaming|wireless|usb|bluetooth|optical|ergonomic|vertical|laptop|pc) mouse",
        r"printers?",
    ],
    # ---- NEW in scope: game controllers (consoles themselves stay OUT: game titles carry console names) ----
    "gaming": [
        r"game ?controllers?", r"gamepads?", r"joysticks?", r"joy-?cons?",
        r"(?:ps[2345]|xbox(?: one| 360| series [xs])?|switch|nintendo(?: switch)?|wii(?: u)?|playstation ?\d?|pc|gaming|"
        r"wireless|bluetooth|usb|pro|steam|android|phone|mobile) controllers?",
        r"controllers? for (?:gaming|games|phone|android|iphone|pc)",
    ],
    # ---- NEW in scope: smart plugs / bulbs / switches ----
    "smart_home": [
        r"smart (?:plugs?|bulbs?|switch(?:es)?|outlets?|lights?|lighting)", r"(?:wi-?fi|wifi|zigbee) (?:plugs?|outlets?|switch(?:es)?|bulbs?)",
        r"smart light switch(?:es)?", r"zigbee", r"z-wave", r"philips hue", r"\bkasa\b", r"wemo", r"lifx",
    ],
}
# Devices that remain OUT of scope by default (not requested). Each can be re-admitted with allow_buckets={...}.
# NOTE "consoles": console names are part of video-game titles ("Mario Kart - Nintendo Switch"), so admitting
# them floods the dataset with software. Controllers/headsets for consoles are handled by the in-scope groups.
OOS_BUCKETS = {
    "consoles":   [r"ps[2345]", r"playstation(?: ?\d)?", r"xbox(?: one| 360| series [xs])?", r"nintendo(?: switch)?",
                   r"wii(?: u)?", r"game consoles?", r"steam deck"],
    "projectors": [r"projectors?"],
    "drones":     [r"drones?", r"quadcopters?"],
    "ereaders":   [r"e-?readers?", r"kindle(?! fire)"],
    "scanners":   [r"scanners?"],
}
OOS_RX = {b: re.compile(r"(?<!with\s)(?<!without\s)(?<!for\s)(?<!and\s)(?<!w/)\b(?:" + "|".join(p) + r")\b", re.I)
          for b, p in OOS_BUCKETS.items()}
GAMING_RX = _rx(CATEGORY_TERMS["gaming"])
OOS_MAX_QUERY_WORDS = 6        # long natural-language queries mention peripherals as requirements, not as the product

# ---------------- 2. ACCESSORY VETO (accessory is the head noun) ----------------
_ACC = (r"cases?(?!\s+fans?)|covers?|screen protectors?|protectors?|tempered glass|skins?|sleeves?|"
        r"pouch(?:es)?|holsters?|bags?|backpacks?|stickers?|decals?|tripods?|lanyards?|wall mounts?|"
        r"mounts?|brackets?|stands?|holders?|straps?|grips?|lens caps?|remote covers?|"
        r"desks?|tables?|carts?|shelf|shelves|chairs?|risers?|remotes?|remote controls?")
ACC_TAIL = re.compile(rf"(?<!with\s)(?<!without\s)(?<!no\s)(?<!and\s)(?<!w/)(?<!charging\s)(?<!for\s)(?<!-)(?<!\be\s)(?<!\brf\s)(?<!\bz\s)(?<!\bf\s)(?<!\bx\s)(?<!\bef\s)(?<!\bl\s)\b(?:{_ACC})(?:\s+[\d.]+(?:\s*(?:pack|pcs|ct|count|inch|in))?)?\s*$", re.I)
ACC_FOR = re.compile(rf"\b(?:{_ACC})\b[^,]{{0,40}}?\b(?:for|compatible with|fits?)\b", re.I)
WATCH_BAND = re.compile(r"\bwatch\s+(?:bands?|straps?)\b|\bbands? for (?:apple |galaxy |smart )?watch", re.I)

# ---------------- 3. HARD VETO (homonyms + the failure classes found in audits) ----------------
HARD_VETO = _rx([
    # original homonyms
    r"food processors?", r"word processors?", r"dodge", r"dodge ram", r"ram (?:\d{3,4}|trucks?|pickup|hitch|promaster|horns?)",
    r"charger challenger", r"as seen on tv", r"tv (?:shows?|series|guide|trays?|dinners?|cabinet)",
    r"writing tablets?", r"emerald tablet", r"fan (?:fiction|fest|club|gear|shop)",
    r"ceiling fans?", r"exhaust fans?", r"hand ?held manual", r"hand fans?", r"folding fans?", r"paper fans?",
    r"vitamins?", r"supplements?", r"capsules?", r"chewables?", r"effervescent", r"melatonin", r"probiotics?",
    r"denture", r"water purification", r"chlorine", r"iodine", r"toys?", r"pretend play", r"dolls?", r"costumes?",
    r"ink", r"toners?", r"cartridges?", r"car battery", r"trickle", r"jump starters?",
    # large appliances
    r"washing machines?", r"washers?", r"dryers?", r"refrigerators?", r"fridges?", r"freezers?", r"microwaves?",
    r"ovens?", r"range hoods?", r"air conditioners?", r"vacuum(?: cleaners?)?", r"cooktops?", r"rice cookers?",
    r"air fryers?", r"blenders?", r"dishwasher", r"detergent",
    # audit class: woodworking / outdoor & power tools
    r"router (?:bits?|tables?|lifts?|bases?|collets?|jigs?|planes?|guides?|templates?|sleds?|depth gauges?|replacement)",
    r"cnc routers?", r"wood(?:en|working)? routers?", r"trim routers?", r"plunge routers?", r"laminate trimmers?",
    r"edgers?", r"trimmers?", r"chainsaws?", r"lawn ?mowers?", r"leaf ?blowers?", r"pole ?saws?", r"weed ?wackers?",
    r"pressure washers?", r"power tools?", r"cordless drills?", r"impact drivers?", r"circular saws?", r"jig ?saws?",
    r"angle grinders?",
    # audit class: furniture
    r"(?:computer|laptop|standing|gaming|writing|office|folding) desks?", r"tv stands?", r"entertainment centers?",
    r"media consoles?", r"credenzas?", r"bookcases?", r"dressers?", r"nightstands?", r"sofas?", r"couch",
    # audit class: exercise equipment
    r"exercise bikes?", r"stationary bikes?", r"spin bikes?", r"treadmills?", r"ellipticals?", r"rowing machines?",
    # audit class: fashion / novelty
    r"crossbody", r"wallets?", r"handbags?", r"totes?", r"drawstring", r"leggings", r"t-?shirts?",
    r"hoodies?", r"socks", r"charm straps?", r"phone charms?", r"jewelry",
    # audit class: services / plans, optics, books
    r"sim kits?", r"unlimited talk", r"prepaid (?:plans?|cards?|sim|data|minutes|service|wireless|airtime)", r"data plans?", r"activation kits?", r"wireless (?:service|plans?)",
    r"telescopes?", r"monoculars?", r"binoculars?", r"microscopes?", r"spotting scopes?",
    r"a novel", r"a memoir", r"paperback", r"hardcover", r"audiobooks?",
    # v2.1: case/protector brands, collectibles, tripods, replacement parts, pharma leaks
    r"otterbox", r"spigen", r"ringke", r"supcase", r"caseology", r"zagg", r"urban armor gear", r"lifeproof",
    r"incipio", r"casetify", r"ghostek", r"tech21", r"funko", r"figurines?", r"action figures?", r"collectibles?",
    r"tripods?", r"selfie sticks?", r"monopods?", r"(?:battery|screen|lcd|digitizer|display) replacement",
    r"replacement (?:batter(?:y|ies)|screens?|parts?|lcd)", r"repair (?:kits?|tools?)",
    r"mg", r"allergy", r"antihistamine", r"ibuprofen", r"acetaminophen", r"aspirin", r"antacid", r"medicine",
    r"acid reducer", r"niacinamide", r"loratadine", r"cetirizine", r"multivitamins?", r"biotin", r"zinc",
    # v2.2: more case brands, parts, consumables, lighting accessories
    r"speck", r"pelican", r"rhinoshield", r"nillkin", r"case-?mate", r"poetic", r"ulak", r"miracase", r"zizo",
    r"tudia", r"fintie", r"procase", r"jetech", r"smartish",
    r"battery (?:cover|door)s?", r"back (?:cover|glass)", r"glass back", r"housing", r"flashlights?", r"ring lights?",
    r"selfie lights?", r"instax[\w ]{0,12}films?", r"(?:printer|photo|copy|inkjet|glossy|label|sticker) paper",
    r"filaments?", r"printer (?:ink|ribbons?|labels?|cartridges?|toners?|rolls?)",
    # new-scope false-positive guards: pests, medical/baby monitors, musical keyboards, mouse pads, lens eyewear
    r"mouse (?:traps?|pads?|ears?|mat)", r"mice (?:traps?|repellent)", r"(?:rat|mouse|mice) (?:repellent|poison|bait|control)",
    r"mickey mouse", r"minnie mouse", r"mousepads?",
    r"(?:baby|heart rate|blood pressure|glucose|fetal|sleep|pulse|breathing) monitors?",
    r"monitor (?:arms?|lights?|light bars?|risers?|stands?|mounts?|hoods?|shades?)",
    r"(?:piano|music|musical|midi|casio|yamaha) keyboards?", r"keyboard (?:stands?|benches?|bags?|covers?|cats?)",
    r"contact lens(?:es)?", r"eyeglass(?:es)?", r"sunglass(?:es)?", r"reading glasses", r"prescription", r"magnifying",
    r"lens (?:cloths?|wipes?|cleaners?|pens?|cleaning|hoods?|filters?|caps?|adapter rings?|cases?)",
    r"switch (?:plates?|covers?)", r"outlet covers?", r"wall plates?",
    r"tpms", r"doppler", r"fetal", r"heartbeat monitors?",
    r"(?:tire|tyre|pressure|temperature|humidity|air quality|co2|radon|energy|pet|dog|cat|bed|wet|moisture|aquarium|fish tank) monitors?",
    r"microphone (?:stands?|arms?)", r"mic (?:stands?|arms?)", r"pop filters?", r"windscreens?", r"boom arms?",
    r"(?:ps[2345]|xbox(?: one| 360)?|switch|nintendo(?: switch)?|wii|video|board|card|party) games?(?!\s*(?:controllers?|consoles?|pads?|sticks?))",
])

# ---- QUERY-ONLY vetoes (too aggressive for product titles: "AirPods Pro with MagSafe Charging Case",
#      "Fitbit Charge 4 (S & L Bands Included)", "Laptop ... Backlit Keyboard" are real electronics)
CASE_ANY = re.compile(r"(?<!charging\s)(?<!without\s)(?<!no\s)\bcases?\b", re.I)
PC_CASE_OK = re.compile(r"\b(?:pc|computer|gaming|tower|atx|itx|server|open|mid[- ]tower|full[- ]tower)\s+cases?\b|"
                        r"\bcases?\s+fans?\b|\bcases? (?:for|with) (?:a )?(?:gaming )?(?:pc|computer)\b", re.I)
BAG_ANY = re.compile(r"\b(?:backpacks?|bags?|sleeves?|briefcases?|messenger|luggage)\b", re.I)
DEV_ACC = re.compile(rf"\b(?:phones?|iphones?|smart ?phones?|laptops?|tablets?|ipads?|cameras?|tvs?|airpods?|watch(?:es)?|macbooks?)\s+(?:{_ACC})\b", re.I)
WATCH_BAND2 = re.compile(r"\b(?:watch(?:es)?|fitbit|garmin|amazfit|versa|alta|charge \d|inspire|forerunner|vivoactive|vivosmart)\b[^,]{0,40}?(?<!without\s)(?<!no\s)\b(?:bands?|straps?|wristbands?)\b", re.I)
# ---------------- 4. BRANDS (supporting evidence only) ----------------
BRAND_ALLOWLIST = [
    "samsung", "apple", "xiaomi", "sony", "lg", "dell", "hp", "lenovo", "asus", "acer", "panasonic", "canon",
    "nikon", "gopro", "logitech", "jbl", "bose", "anker", "tp-link", "netgear", "corsair", "kingston",
    "western digital", "seagate", "oneplus", "oppo", "vivo", "realme", "huawei", "honor", "google", "motorola",
    "nokia", "tecno", "infinix", "poco", "redmi", "nothing", "fitbit", "garmin", "amazfit", "zte", "microsoft",
    "msi", "razer", "toshiba", "sandisk", "crucial", "adata", "gigabyte", "intel", "amd", "nvidia", "evga",
    "hyperx", "g.skill", "teamgroup", "pny", "wd", "sennheiser", "beats", "skullcandy", "jabra", "soundcore",
    "marshall", "harman kardon", "bang & olufsen", "ultimate ears", "edifier", "creative", "audio-technica",
    "ugreen", "baseus", "belkin", "aukey", "d-link", "linksys", "ubiquiti", "tenda", "mercusys", "cable matters",
    "amazon basics", "dji", "insta360", "arlo", "ring", "wyze", "eufy", "reolink", "hikvision", "dahua",
    "fujifilm", "olympus", "tcl", "hisense", "vizio", "sharp", "philips", "skyworth",
]
BRAND_SET = set(BRAND_ALLOWLIST)
BRAND_RX = _rx([re.escape(b) for b in BRAND_ALLOWLIST])
CONGLOMERATE = {"samsung", "sony", "lg", "panasonic", "philips", "sharp", "toshiba"}
HW_NOUNS = _rx([r"players?", r"consoles?", r"headphones?", r"speakers?", r"cameras?", r"batter(?:y|ies)",
                r"receivers?", r"soundbars?", r"radios?", r"earbuds?", r"earphones?", r"turntables?", r"projectors?",
                r"tvs?", r"televisions?", r"phones?", r"laptops?", r"tablets?", r"watch(?:es)?", r"memory cards?"])
_SPEC = re.compile(r"^\d+(?:\.\d+)?(?:gb|tb|mb|mah|w|hz|inch|in|ft|mm|cm|kg|lb|lbs|oz|ml|pack|pcs|ct|k|mp|v|a|ghz|fps)$", re.I)

def _has_model_token(text):
    for tok in re.findall(r"[a-z0-9][a-z0-9-]{2,}", text.lower()):
        if re.search(r"[a-z]", tok) and re.search(r"\d", tok) and not _SPEC.match(tok):
            return True
    return False

def build_category_rx(allow_buckets=()):
    """Category regexes. Buckets in `allow_buckets` (subset of OOS_BUCKETS) are re-admitted as extra groups."""
    terms = dict(CATEGORY_TERMS)
    for b in allow_buckets:
        terms["oos_" + b] = OOS_BUCKETS[b]
    return {g: _rx(p) for g, p in terms.items()}

def veto_reason(text, is_query=False, allow_buckets=()):
    """None, or why the text is excluded: 'hard' | 'accessory' | 'oos:<bucket>'."""
    if HARD_VETO.search(text):
        return "hard"
    if ACC_TAIL.search(text) or ACC_FOR.search(text) or WATCH_BAND.search(text):
        return "accessory"
    if is_query:
        if CASE_ANY.search(text) and not PC_CASE_OK.search(text):
            return "accessory"
        if DEV_ACC.search(text) or WATCH_BAND2.search(text) or BAG_ANY.search(text):
            return "accessory"
        if len(text.split()) <= OOS_MAX_QUERY_WORDS:
            for b, rx in OOS_RX.items():
                if b in allow_buckets or not rx.search(text):
                    continue
                if b == "consoles" and GAMING_RX.search(text):      # "xbox controller usb cable" stays in scope
                    continue
                return "oos:" + b
    return None

def is_vetoed(text, is_query=False, allow_buckets=()):
    return veto_reason(text, is_query, allow_buckets) is not None

def match_group(text, crx, brand=None, is_query=False, allow_buckets=(), use_brand_route=False):
    """Electronics group of a product title (brand=product_brand) or of a query (is_query=True), else None.
    The brand+model route is OFF by default: it admitted printers, marine GPS etc. (brand makes other things)."""
    if not isinstance(text, str) or not text.strip() or is_vetoed(text, is_query, allow_buckets):
        return None
    for g, rx in crx.items():
        if rx.search(text):
            return g
    if use_brand_route:
        if brand is None:
            m = BRAND_RX.search(text)
            brand = m.group(0) if m else ""
        b = str(brand).lower().strip()
        if b in BRAND_SET and _has_model_token(text) and (b not in CONGLOMERATE or HW_NOUNS.search(text)):
            return "brand_model"
    return None

In [ ]:
crx = build_category_rx()

# (text, brand, is_query, should_keep)
CASES = [
 # --- v2.2 scope additions: MUST be kept
 ("gaming monitor 144hz 27 inch", None, True, True), ("27 computer monitor", None, True, True), ("32 inch monitor", None, True, True),
 ("mechanical keyboard", None, True, True), ("wireless mouse", None, True, True), ("wired keyboard for ipad", None, True, True),
 ("hp printer", None, True, True), ("portable printer for phone", None, True, True), ("11x17 hp printer", None, True, True),
 ("HP OfficeJet Pro 9015 Wireless All-in-One Printer", "hp", False, True),
 ("xbox controller usb cable", None, True, True), ("ps4 controller", None, True, True), ("wii controller", None, True, True),
 ("android phone controller for gaming", None, True, True), ("mobile game controller", None, True, True),
 ("Xbox Wireless Controller - Carbon Black", "microsoft", False, True),
 ("usb microphone", None, True, True), ("razer microphone seiren x", None, True, True), ("Blue Yeti USB Microphone", "blue", False, True),
 ("canon 50mm lens", None, True, True), ("Sigma 85mm f1.4 Art Lens for Sony E-mount", "sigma", False, True),
 ("smart plug", None, True, True), ("wifi smart bulbs", None, True, True), ("kasa light switch", None, True, True),
 ("TP-Link Kasa Smart Wi-Fi Plug Mini", "tp-link", False, True), ("TP-Link Kasa Smart Light Switch Works with Alexa", "tp-link", False, True),
 # --- over-veto fixes from the last audit table
 ("headset for ps4", None, True, True), ("wireless earbuds with microphone for iphone", None, True, True),
 ("computer headset with microphone", None, True, True), ("airpods with wireless charging case", None, True, True),
 ("Solid state drive, fast, with copious memory. Two USB ports minimum. High resolution display. Prefer 12” monitor or larger, and prefer number pad.", None, True, True),
 # --- earlier controls
 ("Apple AirPods Pro (2nd Generation) with MagSafe Charging Case", "apple", False, True),
 ("Fitbit Charge 4 Fitness Tracker GPS (S & L Bands Included)", "fitbit", False, True),
 ("ASUS Laptop 15.6 Backlit Keyboard Windows 11", "asus", False, True), ("Samsung Galaxy Tab A8 10.5 inch Android Tablet", "samsung", False, True),
 ("TP-Link AC1750 Smart WiFi Router Archer A7", "tp-link", False, True), ("samsung wireless earbuds", None, True, True),
 ("gaming phone good battery", None, True, True), ("open case gaming pc", None, True, True), ("echo dot 4th gen", None, True, True),
 # --- false-positive guards for the NEW scope: MUST be dropped
 ("baby monitors without wifi", None, True, False), ("video baby monitor", None, True, False), ("heart rate monitor", None, True, False),
 ("piano keyboard for scholl", None, True, False), ("music stand for keyboard", None, True, False), ("computer mouse pad", None, True, False),
 ("mouse trap", None, True, False), ("mickey mouse", None, True, False), ("contact lenses", None, True, False),
 ("camera lens cleaning kit", None, True, False), ("HP 63XL Ink Cartridge", "hp", False, False), ("3d printer filament", None, True, False),
 ("monitor stand", None, True, False), ("monitor arm dual", None, True, False), ("ps4 games", None, True, False),
 ("Mario Kart 8 Deluxe - Nintendo Switch", "nintendo", False, False), ("decorative light switch plate", None, True, False),
 ("microphone stand", None, True, False), ("pop filter", None, True, False),
 # --- buckets that stay OUT
 ("nintendo switch bluetooth", None, True, False), ("pocket projector", None, True, False), ("camera drone for adults", None, True, False),
 ("kindle paperwhite", None, True, False), ("wii charging station", None, True, False), ("document scanner", None, True, False),
 # --- earlier fixes
 ("ring light for phone", None, True, False), ("streamlight flashlight charger", None, True, False),
 ("galaxy s7 edge g935 glass back battery cover door", None, True, False), ("pelican clear iphone xr", None, True, False),
 ("iphone 11 speck", None, True, False), ("fugi film instax film", None, True, False), ("oneplus 7 pro case kickstand", None, True, False),
 ("galaxy watch active band", None, True, False), ("tripod smartphone", None, True, False),
 ("iphone 11 pro case otterbox symmetry", None, True, False), ("taylor swift", None, True, False), ("vizio tv stand", None, True, False),
 ("Cooper Desk PRO Folding Laptop Desk", None, False, False), ("Equate Allergy Relief Loratadine Tablets 10 mg 300 Count", "equate", False, False),
 ("RAM Golf Ultimate Hard-Sided Travel Cover", "ram golf", False, False),
 # --- v2.3: over-veto fixes found in the v2.2 tables (must be kept) ---
 ("apple watch without band", None, True, True), ("airpods without case", None, True, True),
 ("prepaid iphone", None, True, True), ("at&t prepaid phone", None, True, True), ("keychain usb flash drive", None, True, True),
 ("portable charger on a keychain", None, True, True), ("purse size power bank for iphone", None, True, True),
 ("body camera with audio recording wearable necklace", None, True, True), ("desk fan with usb plug", None, True, True),
 # --- v2.3: leaks found in the v2.2 tables (must be dropped) ---
 ("high sierra backpack with laptop compartment", None, True, False), ("Autel TPMS Tire Pressure Monitor Programming Tool", None, False, False),
 ("eco doppler fetal", None, True, False), ("Fetal Doppler Baby Heartbeat Monitor", None, False, False),
 ("laptop sleeve 15 inch", None, True, False),
]
NOT_VETOED = ["fan for desk", "fire hd 8 7th generation tablet keyboard without case"]   # no category word, but must not be vetoed
fails = 0
for text, brand, is_q, keep in CASES:
    g = match_group(text, crx, brand, is_query=is_q)
    if (g is not None) != keep:
        fails += 1; print("FAIL (should", "keep" if keep else "drop", "):", text[:80], "->", g, "|", veto_reason(text, is_q))
print(f"{len(CASES)} cases, failures: {fails}")

for t in NOT_VETOED:
    if veto_reason(t, True):
        fails += 1; print("FAIL (over-veto):", t, "->", veto_reason(t, True))
print("not-vetoed checks done; total failures:", fails)

# the allow_buckets knob: re-admit projectors without touching anything else
crx_p = build_category_rx(allow_buckets=("projectors",))
print("allow_buckets knob works:", match_group("pocket projector", crx_p, None, True, allow_buckets=("projectors",)) is not None)

94 cases, failures: 0
not-vetoed checks done; total failures: 0
allow_buckets knob works: True


### Recorded results — Step 3, first run (v2.0 definitions, thresholds a=0.30 / b=0.60)
* v2 kept **6,891 of 97,345** US queries (old filter: 17,919). Overlap 6,511; old-only 11,408; v2-only 380.
* 5,036 queries contain an electronics term. Their share-of-electronic-E/S-products is **bimodal**: 2,940 sit in (0.9, 1.0], 3,476 are ≥ 0.8; only 706 are < 0.3.
* Rule B (no term in query) contributed 2,559 of the 6,891 kept queries at b = 0.6 — the main source of leaks.
* Kept lists are full-depth again: mean 17.7, median 16 judged products; 132 kept queries have > 40.
* Evaluation sizes (kept queries): test 1,567 (small 627, non-small 940); train 5,324.
* Reading the tables found these leak classes → fixed in v2.1: bare `tablet(s)` (pharma), bare `alexa` (smart-home), bare `doorbell`, `<device> case <modifier>` and case-brand queries (Otterbox/Spigen), watch/fitness bands, replacement batteries, Funko `TV:` titles, tripods/selfie sticks, plus out-of-scope devices (printers, game controllers, mice/keyboards, lenses, smart plugs/bulbs) entering through "works with phone/Alexa" product text.
* The brand + model-number route admitted printers and marine GPS → **switched off** (the category words already catch the real electronics).

### Recorded results — Step 3, second run (v2.1 definitions, thresholds a=0.50 / b=0.80)
* 58,110 electronics-looking products (old filter 60,664). **4,820 of 97,345** US queries kept (5.0 %); Rule A 3,521 · Rule B 1,299. Overlap with the old filter 4,672; old-only 13,247 (5,309 vetoed, 326 term-but-low-share, 7,612 no-term-low-share); v2.1-only 148.
* Rule A share cutoff barely matters (3,655 kept at 0.3 vs 3,521 at 0.5). Rule B is sensitive (1,936 at 0.6 · 1,555 at 0.7 · 1,299 at 0.8 · 891 at 0.9).
* Full lists intact: mean 17.77, median 16; 102 kept queries have > 40 judged products. Label mix E 67.2 / S 22.7 / C 4.5 / I 5.6 %.
* Evaluation sizes then: test 1,086 (small 416) · train 3,734. Sources: other 4,519 · parse_pattern 98 · negations 93 · nlqec 85 · behavioral 25.
* Hand-read of 3E tables (not a formal audit): Rule B ≈ 94 % electronics (48/50); Rule A at share 0.5-0.7 ≈ 83 % (≈5/30 leaks); v2.1-only ≈ 80 %.
* Over-veto bugs found → fixed in v2.2: peripheral/microphone words used as *modifiers* ("headset **with microphone**", "computer **without monitor**", "headset **for ps4**"), long natural-language queries, "charging case" (AirPods), `E-mount` read as an accessory mount, parts/film/flashlight/ring-light leaks, extra case brands.
* **Scope decision (user):** printers, game controllers, mice/keyboards, monitors, microphones, camera lenses and smart plugs/bulbs/switches are now **in scope** (v2.2). Game consoles stay out: game titles carry console names ("Mario Kart – Nintendo Switch") and would flood the set with software.

### Recorded results — Step 3, third run (v2.2 definitions, thresholds a=0.50 / b=0.80)
* 63,614 electronics-looking products. **5,647 of 97,345** US queries kept (5.8 %); Rule A 4,177 · Rule B 1,470. Overlap with old filter 5,329; old-only 12,590; v2.2-only 318.
* Veto reasons over all US queries: hard 8,116 · accessory 6,279 · out-of-scope bucket 647. Remaining excluded buckets cost little: consoles 103, projectors 56, drones 25, scanners 4, e-readers 2 (queries with ≥ 3 E/S products and ≥ 50 % electronic share) → **left out**.
* Kept queries by scope group (A+B): mobile_computing 1,812 · components_accessories 1,156 · home_electronics 816 · audio 640 · computer_peripherals 494 · imaging 613 · gaming 64 · smart_home 52.
* Full lists intact: mean 17.80, median 16; 118 kept queries > 40 judged products. Label mix E 67.0 / S 22.9 / C 4.5 / I 5.6 %.
* Evaluation sizes: **test 1,293 (small 503, non-small 790)** · train 4,354 (small 1,182). Sources: other 5,263 · negations 131 · parse_pattern 130 · nlqec 93 · behavioral 30.
* Hand-read (development samples, *not* the audit): new-scope groups ≈ 95 % correct (2 leaks in 40: tire-pressure monitor tool, fetal doppler); Rule A mid-band ≈ 87 %; over-veto ≈ 8 of 50 high-share vetoed queries (prepaid phones, keychain USB drives, "fan for desk", "… without band/case", body camera "necklace").
* All of these are fixed in **v2.3 (final patch)**; v2.3 is frozen before Step 4.

## Step 3 — Apply the v2 query-level filter to ALL US queries and compare with the old filter
No thresholds are final yet; this step prints the data needed to *choose* them (Rule A cutoff, Rule B cutoff) and the samples to read.
Needs variables from Step 1/2 (`us_examples`, `us_products`, `q_raw`, `sources`, `old_ex`). If the kernel is still open, run only the cells from here down.

In [ ]:
# 3A. product-level v2.3 flags (brand + title), unique US products   [title mode: no query-only vetoes]
crx = build_category_rx()
prod = us_products[["product_id", "product_title", "product_brand"]].copy()
prod["prod_group"] = [match_group(t, crx, b) for t, b in zip(prod["product_title"], prod["product_brand"].fillna(""))]
prod["prod_elec"] = prod["prod_group"].notna()

print("v2.3 electronics-looking products:", int(prod["prod_elec"].sum()), "| old filter:", len(old_ids))
print(prod["prod_group"].value_counts(dropna=False))
print(pd.crosstab(prod["product_id"].isin(old_ids).rename("old_filter"), prod["prod_elec"].rename("v2_filter")))

v2.3 electronics-looking products: 63984 | old filter: 60657
prod_group
None                      1151870
mobile_computing            25225
components_accessories      13273
home_electronics             7732
imaging                      6726
audio                        5346
computer_peripherals         4203
gaming                        830
smart_home                    649
Name: count, dtype: int64
v2_filter     False  True 
old_filter                
False       1129991  25206
True          21879  38778


In [ ]:
# 3B. one row per US query: query-text match + veto reason + electronics share among its Exact/Substitute products
ex = us_examples[["query_id", "query", "product_id", "esci_label"]].merge(
    prod[["product_id", "prod_group", "prod_elec"]], on="product_id", how="left")
ex["prod_elec"] = ex["prod_elec"].fillna(False).astype(bool)

es = ex[ex["esci_label"].isin(["E", "S"])]
g_es  = es.groupby("query_id").agg(n_es=("prod_elec", "size"), share_es=("prod_elec", "mean"))
top   = es[es["prod_elec"]].groupby("query_id")["prod_group"].agg(lambda s: s.mode().iat[0]).rename("top_prod_group")

q = (us_examples.drop_duplicates("query_id")[["query_id", "query"]].set_index("query_id")
     .join(q_raw[["split", "small", "n_judged"]])
     .join(sources.set_index("query_id"))
     .join(g_es).join(top))
q["query_group"]  = [match_group(t, crx, is_query=True) for t in q["query"]]
q["veto_reason"]  = [veto_reason(t, is_query=True) for t in q["query"]]
q["query_vetoed"] = q["veto_reason"].notna()
q["n_es"] = q["n_es"].fillna(0).astype(int)
q["share_es"] = q["share_es"].fillna(0.0)
q["old_keep"] = q.index.isin(old_ex["query_id"].unique())
print(len(q), "US queries |", int(q["query_group"].notna().sum()), "have an electronics term |", int(q["query_vetoed"].sum()), "vetoed")
print("\nveto reasons:\n", q["veto_reason"].str.split(":").str[0].value_counts())
print("\nremaining out-of-scope buckets -- queries with >= 3 E/S products and >= 50% electronic E/S products that they cost:")
print(q[q["veto_reason"].fillna("").str.startswith("oos:") & (q["n_es"] >= 3) & (q["share_es"] >= 0.5)]["veto_reason"].value_counts())

97345 US queries | 4735 have an electronics term | 14817 vetoed

veto reasons:
 veto_reason
hard         7355
accessory    6815
oos           647
Name: count, dtype: int64

remaining out-of-scope buckets -- queries with >= 3 E/S products and >= 50% electronic E/S products that they cost:
veto_reason
oos:consoles      103
oos:projectors     56
oos:drones         25
oos:scanners        4
oos:ereaders        2
Name: count, dtype: int64


In [ ]:
# 3C. thresholds
has_term = q["query_group"].notna() & ~q["query_vetoed"]
print("Rule A candidates (query has a term, not vetoed): share of E/S products that look electronic")
print(pd.cut(q.loc[has_term, "share_es"], [-0.01, 0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]).value_counts().sort_index())
print("\nRule A kept at cutoff a:")
print(pd.Series({a: int((has_term & (q["share_es"] >= a)).sum()) for a in [0.3, 0.4, 0.5, 0.6, 0.7]}))
print("\nRule B kept at cutoff b (no term, not vetoed, n_es >= 3):")
noterm = ~has_term & ~q["query_vetoed"] & (q["n_es"] >= 3)
print(pd.Series({b: int((noterm & (q["share_es"] >= b)).sum()) for b in [0.6, 0.7, 0.8, 0.9]}))

def keep_rule(q, a=0.50, b=0.80, n_min=3):
    ok = ~q["query_vetoed"]
    A = q["query_group"].notna() & (q["share_es"] >= a)
    B = q["query_group"].isna() & (q["share_es"] >= b) & (q["n_es"] >= n_min)
    return (A | B) & ok, A & ok, B & ok

q["v2_keep"], q["rule_A"], q["rule_B"] = keep_rule(q)       # PROVISIONAL a=0.50, b=0.80 (frozen after Step 4)
print("\nv2.2: kept", int(q["v2_keep"].sum()), "of", len(q), "| Rule A:", int(q["rule_A"].sum()), "| Rule B:", int(q["rule_B"].sum()))
print(pd.crosstab(q["old_keep"].rename("old_keep"), q["v2_keep"].rename("v2_keep")))

Rule A candidates (query has a term, not vetoed): share of E/S products that look electronic
share_es
(-0.01, 0.0]     108
(0.0, 0.1]        67
(0.1, 0.2]       123
(0.2, 0.3]        69
(0.3, 0.4]       111
(0.4, 0.5]       147
(0.5, 0.6]       139
(0.6, 0.7]       262
(0.7, 0.8]       304
(0.8, 0.9]       670
(0.9, 1.0]      2735
Name: count, dtype: int64

Rule A kept at cutoff a:
0.3    4371
0.4    4264
0.5    4189
0.6    3997
0.7    3720
dtype: int64

Rule B kept at cutoff b (no term, not vetoed, n_es >= 3):
0.6    2196
0.7    1768
0.8    1476
0.9    1021
dtype: int64

v2.2: kept 5665 of 97345 | Rule A: 4189 | Rule B: 1476
v2_keep   False  True 
old_keep              
False     79120    308
True      12560   5357


In [ ]:
# 3D. what v2.2 keeps looks like (full ESCI lists are retained by design)
kept = q[q["v2_keep"]].copy()
kept["scope_group"] = kept["query_group"].fillna(kept["top_prod_group"])
print("judged products per kept query:\n", kept["n_judged"].describe().round(2))
print("kept queries with > 40 judged products (ESCI Task-1 cap):", int((kept["n_judged"] > 40).sum()))
rows = us_examples[us_examples["query_id"].isin(kept.index)]
print("\nlabel mix over the kept queries' full lists:\n", rows["esci_label"].value_counts(normalize=True).round(3))
print("\nkept queries by ESCI split x small_version (evaluation sizes):")
print(pd.crosstab(kept["split"], kept["small"]))
print("\nkept queries by source:\n", kept["source"].value_counts())
print("\nkept queries by scope group x rule:")
print(pd.crosstab(kept["scope_group"], kept["rule_B"].map({True: "Rule B", False: "Rule A"})))

judged products per kept query:
 count    5665.00
mean       17.78
std         8.69
min         8.00
25%        16.00
50%        16.00
75%        16.00
max       198.00
Name: n_judged, dtype: float64
kept queries with > 40 judged products (ESCI Task-1 cap): 118

label mix over the kept queries' full lists:
 esci_label
E    0.669
S    0.230
I    0.056
C    0.045
Name: proportion, dtype: float64

kept queries by ESCI split x small_version (evaluation sizes):
small     0     1
split            
test    788   507
train  3180  1190

kept queries by source:
 source
other            5276
negations         135
parse_pattern     131
nlqec              93
behavioral         30
Name: count, dtype: int64

kept queries by scope group x rule:
rule_B                  Rule A  Rule B
scope_group                           
audio                      482     162
components_accessories     873     287
computer_peripherals       365     120
gaming                      59       5
home_electronics           

In [ ]:
# 3E. read these (last patch round)
pd.set_option("display.width", 200)
def show(mask, n, title, cols=("query", "query_group", "n_es", "share_es"), seed=3):
    s = q[mask].sample(min(n, int(mask.sum())), random_state=seed)
    print(f"\n=== {title}  ({int(mask.sum())} queries) ===")
    print(s[list(cols)].assign(share_es=s["share_es"].round(2)).to_string())

new_groups = {"computer_peripherals", "gaming", "smart_home"}
q["scope_group"] = q["query_group"].fillna(q["top_prod_group"])
show(q["v2_keep"] & q["scope_group"].isin(new_groups), 40, "NEW-scope groups kept (monitors/keyboards/mice/printers, controllers, smart home)")
show(q["query_vetoed"] & ~q["veto_reason"].fillna("").str.startswith("oos:") & (q["share_es"] >= 0.5) & (q["n_es"] >= 3), 50,
     "DROPPED by hard/accessory veto although >= 50% of E/S products look electronic", cols=("query", "veto_reason", "n_es", "share_es"))
show(q["rule_A"] & (q["share_es"] < 0.7), 30, "kept via Rule A, share 0.5-0.7")
q.to_csv("v2_query_table.csv")


=== NEW-scope groups kept (monitors/keyboards/mice/printers, controllers, smart home)  (600 queries) ===
                                                                                         query           query_group  n_es  share_es
query_id                                                                                                                            
10864                                                                    asus predator monitor  computer_peripherals    16      0.94
9514                                                          aoc agon ag352ucg6 black edition                   NaN    16      0.94
4630                                                                           50 inch monitor  computer_peripherals    11      0.91
2693                                                                            2 tray printer  computer_peripherals    15      0.93
114232                                                         xbox 360 controller wired white  

## Step 4 — Freeze v2.3 and run the audit protocol
**Protocol (do not deviate):** (1) the filter and thresholds below are frozen — nothing is patched after this point; (2) the audit sample is drawn with a new seed and **excludes every query that appeared in a development table**; (3) you label **blind** (the file shows only the query and example product titles, not the filter's decision); (4) precision and recall are computed with stratified estimators and bootstrap 95 % CIs; (5) residual errors are *reported*, not fixed.
Cells 3E above are development-only — skip them.

In [ ]:
import numpy as np

FILTER_VERSION = "v2.3"
A_CUT, B_CUT, N_MIN = 0.50, 0.80, 3                                 # FROZEN
q["v2_keep"], q["rule_A"], q["rule_B"] = keep_rule(q, A_CUT, B_CUT, N_MIN)
q["scope_group"] = q["query_group"].fillna(q["top_prod_group"])
print(FILTER_VERSION, "frozen:", int(q["v2_keep"].sum()), "queries kept | Rule A", int(q["rule_A"].sum()), "| Rule B", int(q["rule_B"].sum()))
q.assign(filter_version=FILTER_VERSION).to_csv(f"query_table_{FILTER_VERSION}.csv")

# every query id that was displayed in a development table (so the audit is on genuinely unseen queries)
DEV_SEEN_IDS = {833, 1203, 1309, 1505, 1743, 1747, 1792, 2079, 2382, 2693, 2703, 3022, 3034, 3141, 3355, 3496, 3678, 4222, 4348, 4630, 5046, 5060, 5735, 5750, 6847, 7483, 7492, 7522, 7540, 7565, 7583, 7586, 7587, 7687, 7883, 7901, 8027, 8369, 8729, 9059, 9267, 9692, 9785, 9811, 9848, 9921, 9922, 10297, 10436, 10839, 10924, 11259, 11521, 11602, 11913, 12075, 12433, 12915, 13907, 14380, 14445, 15934, 17011, 17012, 17605, 17889, 18381, 18974, 19294, 19547, 20878, 21182, 21885, 21895, 21906, 22391, 22469, 22494, 22495, 22729, 22742, 23393, 23571, 23793, 23928, 24947, 26902, 27844, 28355, 28363, 28365, 28377, 28403, 28720, 29226, 29253, 29259, 29260, 31032, 31823, 32048, 32162, 32239, 32542, 32606, 32816, 33021, 33227, 33554, 33875, 34657, 34884, 35601, 36129, 36628, 36960, 37206, 37643, 37884, 38741, 39536, 39547, 40454, 40706, 40796, 40815, 40818, 40821, 40824, 40829, 40830, 40832, 40833, 40835, 40864, 40888, 40897, 41295, 42079, 42124, 42720, 42933, 42991, 43013, 43019, 43969, 44005, 44014, 44026, 44067, 44072, 44251, 44271, 44273, 44294, 44497, 44504, 44519, 44704, 45310, 45407, 46036, 46958, 47249, 48101, 48254, 49550, 49738, 49778, 50536, 51167, 51175, 51183, 51686, 51726, 51747, 51854, 51876, 52193, 52310, 52574, 53850, 53872, 53993, 54024, 54055, 54066, 54112, 54158, 54198, 54213, 54233, 54490, 54505, 54510, 54593, 56378, 56748, 56997, 57448, 57466, 57482, 57501, 57573, 57703, 57721, 57807, 58301, 58617, 59021, 59179, 59955, 60132, 60222, 61129, 61321, 61502, 61777, 61927, 63294, 63307, 63679, 64523, 66766, 67097, 68276, 68644, 68903, 69436, 69913, 69937, 70386, 70864, 70868, 70881, 70975, 71083, 71205, 71297, 71335, 71697, 71704, 72309, 72982, 73248, 73332, 73866, 73879, 73893, 74158, 74272, 74456, 74480, 74495, 74499, 74617, 74668, 75531, 75704, 75796, 75800, 75849, 77173, 78301, 78338, 78347, 78486, 78635, 79434, 79456, 79572, 79579, 79595, 79602, 79722, 80611, 81667, 81795, 81833, 81926, 81929, 82417, 82483, 82806, 82884, 82937, 83134, 83412, 83882, 83940, 84651, 84801, 84844, 85467, 85787, 85803, 85813, 86185, 87745, 88117, 88801, 89112, 89406, 89648, 89956, 89967, 89991, 89996, 90010, 90053, 90123, 90152, 90200, 90231, 90232, 90352, 91153, 91320, 91712, 92138, 93170, 94003, 94615, 94992, 94993, 95017, 95047, 95493, 96071, 96089, 96162, 96575, 96581, 96582, 96623, 96748, 96807, 96809, 97507, 97975, 98507, 100920, 100923, 101780, 101948, 102260, 103017, 103458, 104392, 104471, 104473, 104592, 104596, 105001, 105305, 105482, 105890, 105944, 106235, 106333, 106438, 107195, 107293, 107303, 107363, 107392, 108564, 108809, 109052, 109164, 109211, 109831, 110076, 111141, 111636, 111638, 111814, 111816, 112109, 112160, 112163, 113974, 114206, 114239, 114250, 114355, 114606, 115022, 115695}
print(len(DEV_SEEN_IDS), "development-seen query ids excluded from the audit frame")

v2.3 frozen: 5665 queries kept | Rule A 4189 | Rule B 1476
389 development-seen query ids excluded from the audit frame


In [ ]:
# 4B. stratified audit sample (seed 2026)
frame = q[~q.index.isin(DEV_SEEN_IDS)].copy()
oos = frame["veto_reason"].fillna("").str.startswith("oos:")
has_term = frame["query_group"].notna()
unv = ~frame["query_vetoed"]
conds = [
    frame["rule_A"],
    frame["rule_B"],
    frame["query_vetoed"] & ~oos & (frame["n_es"] >= 3) & (frame["share_es"] >= 0.5),
    frame["query_vetoed"],
    unv & ~frame["v2_keep"] & ((has_term & frame["share_es"].between(0.3, 0.4999)) |
                               (~has_term & (frame["n_es"] >= 3) & frame["share_es"].between(0.5, 0.7999))),
    unv & ~frame["v2_keep"] & (frame["share_es"] >= 0.1),
    unv & ~frame["v2_keep"],
]
names = ["K_A_ruleA", "K_B_ruleB", "D1_vetoed_high_share", "D5_vetoed_other", "D2_near_miss", "D3_low_share", "D4_bulk"]
frame["stratum"] = np.select(conds, names, default="other")
SAMPLE_N = {"K_A_ruleA": 70, "K_B_ruleB": 50, "D1_vetoed_high_share": 40, "D5_vetoed_other": 30,
            "D2_near_miss": 50, "D3_low_share": 30, "D4_bulk": 40}

rng = np.random.default_rng(2026)
pick = []
for st, n in SAMPLE_N.items():
    ids = frame.index[frame["stratum"] == st].to_numpy()
    pick += list(rng.choice(ids, size=min(n, len(ids)), replace=False))
sample = frame.loc[pick].copy()

title_of = us_products.set_index("product_id")["product_title"]
rows = us_examples[us_examples["query_id"].isin(sample.index) & us_examples["esci_label"].isin(["E", "S"])]
rows = rows.sort_values(["query_id", "esci_label"])
ex_titles = rows.groupby("query_id")["product_id"].apply(lambda s: " || ".join(title_of.reindex(s.iloc[:3]).fillna("").str.slice(0, 90)))
sample["example_titles"] = ex_titles.reindex(sample.index).fillna("")

blind = sample.reset_index().sample(frac=1, random_state=7).reset_index(drop=True)      # shuffled
blind["audit_id"] = np.arange(1, len(blind) + 1)
blind[["audit_id", "query", "example_titles"]].assign(in_scope="", note="").to_csv("audit_to_label.csv", index=False)
blind[["audit_id", "query_id", "stratum", "v2_keep", "rule_A", "rule_B", "veto_reason", "n_es", "share_es"]].to_csv("audit_key.csv", index=False)
sizes = pd.concat([frame["stratum"].value_counts().rename("N_h"), sample["stratum"].value_counts().rename("n_h")], axis=1)
sizes.index.name = "stratum"
sizes.to_csv("audit_strata_sizes.csv")
print(sizes)
print("\nwrote audit_to_label.csv (%d rows), audit_key.csv, audit_strata_sizes.csv" % len(blind))

### Labeling guide (apply literally; label **blind**, in the order given)
Open `audit_to_label.csv`, fill the `in_scope` column, save as **`audit_labeled.csv`**.

* **1** = the shopper's main intent is to buy a **device or component** in the project scope: smartphones, laptops, tablets, desktops, smartwatches/fitness trackers · RAM, storage, CPUs/GPUs/motherboards, chargers, cables, power banks, USB hubs/adapters · cameras (incl. lenses, dash/web/security/action cams) · TVs, speakers/soundbars, routers/modems/extenders, electric fans · headphones/earbuds/headsets/microphones · monitors, keyboards, mice, printers · game controllers · smart plugs/bulbs/switches.
* **0** = anything else: accessories-only (cases, covers, screen protectors, bands/straps, stands, mounts, bags, tripods), replacement parts, consumables (ink, paper, film, filament), software/games/apps, phone plans, large appliances, tools, toys, furniture, books, or non-electronics.
* **u** = you genuinely cannot decide (reported separately, excluded from the estimates).
* Use the example product titles only to resolve what an ambiguous query means. Do **not** try to guess what the filter did.
* Optional (recommended): ask your supervisor or a teammate to label 50 of the rows independently → report Cohen's kappa.

In [ ]:
# 4C. RUN AFTER LABELING -> precision / recall (stratified estimators; Jeffreys-prior intervals; sensitivity analyses)
def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; dd = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / dd; h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / dd
    return (max(0.0, c - h), min(1.0, c + h))

lab = pd.read_csv("audit_labeled.csv", dtype=str, keep_default_na=False); lab["audit_id"] = lab["audit_id"].astype(int); key = pd.read_csv("audit_key.csv"); sz = pd.read_csv("audit_strata_sizes.csv", index_col=0)
d0 = lab[["audit_id", "in_scope", "query"]].merge(key, on="audit_id")
d0["y"] = d0["in_scope"].astype(str).str.strip().str.lower().map({"1": 1, "0": 0, "1.0": 1, "0.0": 0})
print("unsure / unlabeled:", int(d0["y"].isna().sum()), "of", len(d0))

def per_stratum(d):
    per = d.groupby("stratum")["y"].agg(k="sum", n="count")
    per["rate"] = per["k"] / per["n"]
    per[["ci_lo", "ci_hi"]] = [wilson(k, n) for k, n in zip(per["k"], per["n"])]
    per["N_h"] = sz["N_h"].reindex(per.index)
    return per

def estimates(per, rates):
    N = per["N_h"]; est = N * rates
    kept = [s for s in per.index if s.startswith("K_")]
    tp = est[kept].sum()
    prec = tp / N[kept].sum(); rec = tp / est.sum()
    return prec, rec, 2 * prec * rec / (prec + rec), est.sum()

d = d0.dropna(subset=["y"]); per = per_stratum(d)
print(per.round(3))
p0, r0, f0, tot = estimates(per, per["rate"])

# interval: draw each stratum rate from Beta(k+0.5, n-k+0.5)  (Jeffreys) -> a 0/40 stratum is NOT treated as exactly zero
rng = np.random.default_rng(1)
boot = np.array([estimates(per, pd.Series({s: rng.beta(per.loc[s, "k"] + 0.5, per.loc[s, "n"] - per.loc[s, "k"] + 0.5)
                                           for s in per.index})) for _ in range(4000)])
lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
print(f"\nFilter {FILTER_VERSION}  (audit frame excludes development-seen queries)")
print(f"precision : {p0:.3f}   95% interval [{lo[0]:.3f}, {hi[0]:.3f}]")
print(f"recall    : {r0:.3f}   95% interval [{lo[1]:.3f}, {hi[1]:.3f}]")
print(f"F1        : {f0:.3f}   95% interval [{lo[2]:.3f}, {hi[2]:.3f}]")
print(f"estimated in-scope queries in the frame: {tot:,.0f}")

# sensitivity 1: the 'unsure' rows
print("\nsensitivity to unsure labels:")
for fill in (1, 0):
    pp = per_stratum(d0.assign(y=d0["y"].fillna(fill)))
    a, b, c, _ = estimates(pp, pp["rate"])
    print(f"  all unsure = {fill}: precision {a:.3f}  recall {b:.3f}")

# sensitivity 2: what we cannot see in the bulk stratum (0 found in 40 -> true rate is only bounded)
print("\nrecall if the unvetoed bulk stratum D4 (N = %d) held a true in-scope rate of:" % int(per.loc["D4_bulk", "N_h"] if "D4_bulk" in per.index else 0))
if "D4_bulk" in per.index:
    for r4 in (0, 0.0025, 0.005, 0.01):
        rr = per["rate"].copy(); rr["D4_bulk"] = r4
        print(f"  {r4:6.2%}: recall {estimates(per, rr)[1]:.3f}")
    keep_s = [s for s in per.index if s not in ("D4_bulk", "D5_vetoed_other")]
    pc = per.loc[keep_s]; print(f"conditional recall among queries with any electronics evidence (excl. D4, D5): {estimates(pc, pc['rate'])[1]:.3f}")

print("\nFALSE POSITIVES among kept queries (read, report, do not patch):")
print(d[(d["stratum"].str.startswith("K_") & (d["y"] == 0))][["query", "stratum"]].to_string())
print("\nMISSED in-scope queries among dropped strata:")
print(d[(~d["stratum"].str.startswith("K_") & (d["y"] == 1))][["query", "stratum"]].to_string())

unsure / unlabeled: 20 of 310
                         k   n   rate  ci_lo  ci_hi    N_h
stratum                                                   
D1_vetoed_high_share   4.0  38  0.105  0.042  0.241    100
D2_near_miss          19.0  41  0.463  0.321  0.613   1252
D3_low_share           2.0  25  0.080  0.022  0.250   3081
D4_bulk                0.0  40  0.000  0.000  0.088  72476
D5_vetoed_other        0.0  30  0.000  0.000  0.114  14601
K_A_ruleA             63.0  68  0.926  0.839  0.968   4039
K_B_ruleB             39.0  48  0.812  0.681  0.898   1407

Filter v2.3  (audit frame excludes development-seen queries)
precision : 0.897   95% interval [0.830, 0.938]
recall    : 0.854   95% interval [0.454, 0.866]
F1        : 0.875   95% interval [0.603, 0.886]
estimated in-scope queries in the frame: 5,722

sensitivity to unsure labels:
  all unsure = 1: precision 0.901  recall 0.774
  all unsure = 0: precision 0.869  recall 0.873

recall if the unvetoed bulk stratum D4 (N = 72476) held a 

### Recorded results — Step 4 audit of filter v2.3 (310 blind, stratified, fresh queries; seed 2026)
| stratum | N | audited (labeled) | in-scope rate (Wilson 95 %) |
|---|---|---|---|
| K_A Rule A (kept) | 4,039 | 68 | 92.6 % (83.9–96.8) |
| K_B Rule B (kept) | 1,407 | 48 | 81.2 % (68.1–89.8) |
| D1 vetoed, high electronic share | 100 | 38 | 10.5 % (4.2–24.1) |
| D2 near-miss (term & share 0.3–0.5, or no term & share 0.5–0.8) | 1,252 | 41 | 46.3 % (32.1–61.3) |
| D3 low share (0.1–0.3) | 3,081 | 25 | 8.0 % (2.2–25.0) |
| D4 bulk (share < 0.1, unvetoed) | 72,476 | 40 | 0 % (0–8.8) |
| D5 other vetoed | 14,601 | 30 | 0 % (0–11.4) |

* Precision **0.897** (bootstrap 0.843–0.946) · recall **0.854** (0.793–0.908, *binomial bootstrap — understates the D4 uncertainty; see 4C*) · F1 0.875. 20 of 310 rows labeled "unsure".
* **Error structure.** False positives are mostly *scope-boundary* cases (car stereo / tweeter adapter, camera dolly, CPU water block, SSD enclosure, TV antenna, Apple Pencil, mic preamp, Roku) and a few non-electronics (`mac os book`, `polaroid book album`). Misses are mostly short generic queries just under the cutoffs (`work headphones`, `jbl powered speakers`, `outdoor fan`, `amazon echo`, `huawei t3 10`), device queries whose E/S lists are dominated by accessories (`apple watch 40mm series 5`) and accessory-vetoed queries that name a device (`earbuds with case for kids`).
* **Decision:** filter v2.3 stays frozen (no re-tuning on audit data). Limitations to state in the report: recall loss concentrates in generic short queries and accessory-dominated device queries; bulk-stratum recall is only bounded.

## Step 5 — Freeze the dataset: final tables, retrieval corpus, dataset card
Decisions implemented here:
* **Queries** = the v2.3 kept set (US, Rule A + Rule B), with ESCI `split`, `small_version`, `source`, `scope_group`, `rule`, `share_es` carried along.
* **Judgments** = the **full** ESCI judged list of every kept query (E/S/C/I), with NDCG gains from the ESCI paper: **E 1.0 · S 0.1 · C 0.01 · I 0.0**. Unjudged products count as 0.
* **Retrieval corpus** = the pooled set of products judged for any kept query (train + test). Every query is answered against the whole pool, so the other queries' products act as natural distractors; non-electronic items that were judged *Irrelevant* for a kept query stay in as hard negatives.
* **Evaluation protocol:** classifier / prompt development on `split == "train"`; the ESCI `test` split is used once, for the final four-condition ablation. Report the full test set and the `small_version` test subset (ESCI's published ranking benchmark) separately.

In [ ]:
import json
FINAL_VERSION = "v2.3"
GAIN = {"E": 1.0, "S": 0.1, "C": 0.01, "I": 0.0}

fq = q[q["v2_keep"]].copy()
fq["scope_group"] = fq["query_group"].fillna(fq["top_prod_group"])
fq["rule"] = np.where(fq["rule_A"], "A", "B")
queries_final = fq[["query", "split", "small", "source", "scope_group", "rule", "n_es", "share_es", "n_judged"]].reset_index()
queries_final["filter_version"] = FINAL_VERSION

judg = us_examples[us_examples["query_id"].isin(fq.index)][
    ["example_id", "query_id", "product_id", "esci_label", "split", "small_version", "large_version"]].copy()
judg["gain"] = judg["esci_label"].map(GAIN)
corpus = us_products[us_products["product_id"].isin(judg["product_id"].unique())].copy()

# ---- integrity checks
assert judg["product_id"].isin(corpus["product_id"]).all(), "judged product missing from corpus"
assert not judg.duplicated(["query_id", "product_id"]).any(), "duplicate (query, product) judgments"
assert judg["gain"].notna().all()
print("queries:", len(queries_final), "| judgment rows:", len(judg), "| corpus products:", len(corpus))
print("every kept query has >= 1 E/S product:", bool((fq["n_es"] >= 1).all()))
print("\nqueries by split x small_version:\n", pd.crosstab(queries_final["split"], queries_final["small"]))
print("\njudgment rows by split:\n", judg["split"].value_counts())
print("\nlabel mix by split:\n", pd.crosstab(judg["split"], judg["esci_label"], normalize="index").round(3))
print("\njudged products per query by split:\n", judg.groupby(["split", "query_id"]).size().groupby("split").describe().round(2))
print("\nqueries by scope group x split:\n", pd.crosstab(queries_final["scope_group"], queries_final["split"]))

queries: 5665 | judgment rows: 100740 | corpus products: 51318
every kept query has >= 1 E/S product: True

queries by split x small_version:
 small     0     1
split            
test    788   507
train  3180  1190

judgment rows by split:
 split
train    77593
test     23147
Name: count, dtype: int64

label mix by split:
 esci_label      C      E      I      S
split                                 
test        0.049  0.629  0.061  0.261
train       0.044  0.681  0.055  0.220

judged products per query by split:
         count   mean   std  min   25%   50%   75%    max
split                                                   
test   1295.0  17.87  7.87  8.0  16.0  16.0  16.0  113.0
train  4370.0  17.76  8.93  8.0  16.0  16.0  16.0  198.0

queries by scope group x split:
 split                   test  train
scope_group                        
audio                    152    492
components_accessories   269    891
computer_peripherals     115    370
gaming                    16     48
hom

In [ ]:
# 5B. corpus statistics (sizing the BM25 / FAISS index; text-field coverage)
n_prod_q = judg.groupby("product_id")["query_id"].nunique()
print("products judged for > 1 kept query:", int((n_prod_q > 1).sum()), "of", len(corpus))
tr_p = set(judg.loc[judg["split"] == "train", "product_id"]); te_p = set(judg.loc[judg["split"] == "test", "product_id"])
print("products judged in BOTH train and test queries:", len(tr_p & te_p))
for col in ["product_title", "product_description", "product_bullet_point", "product_brand", "product_color"]:
    print(f"{col:22s} missing: {corpus[col].isna().mean():6.1%}   median length: {int(corpus[col].fillna('').str.len().median())}")
print("\ncorpus products that the product-level electronics rule flags:",
      int(corpus["product_id"].isin(prod.loc[prod['prod_elec'], 'product_id']).sum()), "of", len(corpus))
for dim in (384, 768):
    print(f"FAISS flat index, {dim}-d float32: {len(corpus) * dim * 4 / 1e6:,.0f} MB   (NFR4 limit 1,000 MB; >= 20,000 items required)")

queries_final.to_parquet(f"esci_electronics_{FINAL_VERSION}_queries.parquet")
judg.to_parquet(f"esci_electronics_{FINAL_VERSION}_judgments.parquet")
corpus.to_parquet(f"esci_electronics_{FINAL_VERSION}_corpus.parquet")
card = {"dataset": "ESCI electronics subset (US)", "filter_version": FINAL_VERSION,
        "thresholds": {"rule_A_share": A_CUT, "rule_B_share": B_CUT, "min_es_products": N_MIN},
        "ndcg_gains": GAIN, "n_queries": int(len(queries_final)), "n_judgments": int(len(judg)), "n_corpus_products": int(len(corpus)),
        "audit": {"n": 310, "seed": 2026, "precision": round(float(p0), 3) if "p0" in globals() else None,
                  "recall": round(float(r0), 3) if "r0" in globals() else None}}
json.dump(card, open(f"esci_electronics_{FINAL_VERSION}_card.json", "w"), indent=2)
print("\nwrote 3 parquet files + card.json")

products judged for > 1 kept query: 15902 of 51318
products judged in BOTH train and test queries: 8568
product_title          missing:   0.0%   median length: 146
product_description    missing:  47.2%   median length: 63
product_bullet_point   missing:   8.9%   median length: 691
product_brand          missing:   0.9%   median length: 6
product_color          missing:  30.4%   median length: 5

corpus products that the product-level electronics rule flags: 42190 of 51318
FAISS flat index, 384-d float32: 79 MB   (NFR4 limit 1,000 MB; >= 20,000 items required)
FAISS flat index, 768-d float32: 158 MB   (NFR4 limit 1,000 MB; >= 20,000 items required)

wrote 3 parquet files + card.json


### Recorded results — Step 4 (final estimator) and Step 5 (frozen dataset v2.3)
**Audit (310 blind, stratified, unseen queries).** Precision **0.897** (95 % interval 0.830–0.938). Recall point estimate **0.854**, but its interval is [0.454, 0.866]: the unvetoed bulk stratum D4 (N = 72,476) produced 0/40 in-scope, so its true rate is only bounded. Recall if the D4 in-scope rate were 0 / 0.25 / 0.5 / 1 % → 0.854 / 0.827 / 0.803 / 0.758. Unsure labels (20 / 310) move precision to 0.869–0.901 and recall to 0.774–0.873. Conditional recall among queries with any electronics evidence = 0.854.
**Reporting sentence:** *precision 0.90 (0.83–0.94); recall 0.85 among queries with electronics evidence; overall recall 0.76–0.85 under assumed bulk in-scope rates of 0–1 %.*

**Frozen dataset v2.3.** 5,665 queries · 100,740 judgments (mean 17.8 per query, median 16) · pooled corpus 51,318 products (42,190 flagged electronics by the product rule; the rest are judged-Irrelevant hard negatives).
Test 1,295 queries (507 small_version) · train 4,370 (1,190 small). Label mix: test E 62.9 / S 26.1 / C 4.9 / I 6.1 % · train E 68.1 / S 22.0 / C 4.4 / I 5.5 %.
Corpus: 15,902 products judged for > 1 query; 8,568 judged in both train and test queries (harmless while no retriever is trained on the judgments). Text fields: title 0 % missing (median 146 chars) · bullet points 8.9 % missing (median 691 chars) · description 47.2 % missing · brand 0.9 % · color 30.4 %. FAISS flat index: 79 MB (384-d) / 158 MB (768-d) ≪ 1 GB; 51,318 ≥ 20,000 items.

## Step 6A — Classifier pool diagnostics (train split of the FROZEN dataset only)
Before sampling the 500 queries we check whether Exact / Semantic / Hybrid are all *available* in ESCI's keyword-style queries, and what the length mix looks like.
The heuristics below are **only for diagnostics** (they are not the labels): price words → Hybrid-like, a model-number token → Exact-like, brand without model → mixed, neither → Semantic-like.

In [ ]:
pool = queries_final[queries_final["split"] == "train"].copy()
pool["n_words"] = pool["query"].str.split().str.len()
print(len(pool), "train queries in the frozen dataset")
print("\nwords per query:\n", pool["n_words"].describe().round(2))
print("\nlength buckets (1-2 / 3-4 / 5-6 / 7+):\n", pd.cut(pool["n_words"], [0, 2, 4, 6, 1000]).value_counts().sort_index())
print("\nsource:\n", pool["source"].value_counts())

PRICE_RX = re.compile(r"(?:\bunder\b|\bbelow\b|\bless than\b|\bcheap\w*\b|\bbudget\b|\baffordable\b|\bdiscount\w*\b|\bon sale\b|[$£€]\s?\d|\b\d+\s?(?:usd|dollars?|bucks)\b)", re.I)
pool["has_price"] = pool["query"].str.contains(PRICE_RX)
pool["has_model"] = [_has_model_token(t) for t in pool["query"]]
pool["has_brand"] = pool["query"].str.contains(BRAND_RX)
pool["heuristic"] = np.select(
    [pool["has_price"], pool["has_model"], pool["has_brand"]],
    ["price (Hybrid-like)", "model token (Exact-like)", "brand only (mixed)"], default="neither (Semantic-like)")
print("\nheuristic cells:\n", pool["heuristic"].value_counts())
print("\nheuristic cell x length bucket:\n", pd.crosstab(pool["heuristic"], pd.cut(pool["n_words"], [0, 2, 4, 6, 1000])))

pd.set_option("display.width", 200)
for h in ["price (Hybrid-like)", "model token (Exact-like)", "brand only (mixed)", "neither (Semantic-like)"]:
    s_ = pool[pool["heuristic"] == h]
    print(f"\n=== {h}: 15 random examples of {len(s_)} ===")
    print(s_.sample(min(15, len(s_)), random_state=5)[["query", "scope_group", "n_words"]].to_string(index=False))
nl = pool[pool["source"] == "nlqec"]
print(f"\n=== nlqec (natural-language source): {len(nl)} train queries, 10 examples ===")
print(nl.sample(min(10, len(nl)), random_state=5)[["query", "n_words"]].to_string(index=False))

4370 train queries in the frozen dataset

words per query:
 count    4370.00
mean        3.66
std         2.47
min         1.00
25%         2.00
50%         3.00
75%         4.00
max        29.00
Name: n_words, dtype: float64

length buckets (1-2 / 3-4 / 5-6 / 7+):
 n_words
(0, 2]       1180
(2, 4]       2301
(4, 6]        678
(6, 1000]     211
Name: count, dtype: int64

source:
 source
other            4086
negations          97
parse_pattern      95
nlqec              74
behavioral         18
Name: count, dtype: int64

heuristic cells:
 heuristic
neither (Semantic-like)     3000
brand only (mixed)           942
model token (Exact-like)     390
price (Hybrid-like)           38
Name: count, dtype: int64

heuristic cell x length bucket:
 n_words                   (0, 2]  (2, 4]  (4, 6]  (6, 1000]
heuristic                                                  
brand only (mixed)           198     536     169         39
model token (Exact-like)     155     173      47         15
neither (Sema

### Recorded results — Step 6A (classifier-pool diagnostics, 4,370 train queries)
* Length: median 3 words, mean 3.66; buckets 1-2 → 1,180 · 3-4 → 2,301 · 5-6 → 678 · 7+ → 211. Sources: other 4,086 · negations 97 · parse_pattern 95 · nlqec 74 · behavioral 18.
* Heuristic cells (diagnostic only): neither / Semantic-like 3,000 · brand only 942 · model token / Exact-like 390 · **price words 38 (0.9 %)**.
* **Finding:** the proposal's Hybrid definition (price constraint + descriptive terms) covers < 1 % of ESCI queries and the price phrases that exist are mostly in the 74 natural-language `nlqec` queries. A price-defined Hybrid class cannot be trained *or* tested from ESCI (a 15 % test split would hold ≈ 6 examples). In addition, price is already extracted separately for M3 (FR9), and α = P(Exact) only needs the Exact-vs-rest probability.
* **Working definition (v6, to be confirmed with the supervisor):** classes are defined by *lexical specificity* (does a keyword match pin down the product?), and the price constraint is recorded as an independent binary flag so the proposal's original Hybrid can still be derived.

## Step 6B — Stratified sample of 550 train queries to label (550 → ≈ 500 usable after `N/A`)
Design: all 74 `nlqec` queries (the only natural-language, need-style queries — closest to the live-user queries the classifier will see) · all non-nlqec price-word queries · the rest stratified by heuristic cell and length bucket with oversampling of long queries (weights 1 / 1 / 1.5 / 2 for 1-2 / 3-4 / 5-6 / 7+ words). Sampling is class-**un**balanced on purpose; report macro-F1, not just accuracy.

### Labeling protocol (apply in this order; label blind)
**`intent`** — one of `Exact`, `Hybrid`, `Semantic`, `N/A`
1. **N/A** — not a device/component query in scope (residual filter errors) or unintelligible. Count them; they are reported, not hidden.
2. **Exact** — the query *pins down* one product (or one tiny family such as colour variants): a model / part code, or product line + number / generation, or a named brand with an exact spec combination. Test: *could a human point to ONE product from this string?*
   `ga503qs-xs98q-wh` · `lg oled77cxpua` · `thinkpad x1 yoga 3rd generation` · `samsung a30 gsm unlocked phone` · `asus chromebook flip c434` · `logitech g935` · `rode sc4` · `motorola moto z4`
3. **Hybrid** — a **brand or product-family anchor** is present but several different products fit (no pinning code); the keyword anchor and the descriptive words both matter. A misspelled brand still counts as a brand.
   `jbl speaker` · `adata ssd` · `amd ryzen` · `nikon zoom lens` · `kingston digital datatraveler flash drive` · `wireless earbuds samsung galaxy buds` · `nubwo gaming headset` · `sabrent 16 port usb hub` · `samsumg bluetooth headphones` · `I would buy a macbook pro … refurbished`
4. **Semantic** — **no** brand / product-line / model anchor: a category, attribute, use-case or need.
   `wireless home theater speakers` · `indoor security camera` · `karaoke machine for adults` · `lightning to hdmi cable` · `vibration sensor connect to phone` · `An everyday laptop for surfing the net and emails and not too expensive.`

**`price`** — `1` only if the query states a price or budget constraint (`under 100`, `$20`, `max budget £400`, `around £400`, `cheap`, `cheapest`, `budget`, `affordable`, `on sale`, `discount`), otherwise `0`. Independent of `intent`.
Optional: have a second person label 50 rows → Cohen's kappa on `intent`.

In [ ]:
N_TARGET_NOTE = 550
rng = np.random.default_rng(2026)
pool["bucket"] = pd.cut(pool["n_words"], [0, 2, 4, 6, 1000], labels=["1-2", "3-4", "5-6", "7+"]).astype(str)
pool["stratum6"] = np.select([pool["source"] == "nlqec", pool["has_price"], pool["has_model"], pool["has_brand"]],
                             ["NL_all", "price_all", "model", "brand_only"], default="neither")
QUOTA = {"NL_all": None, "price_all": None, "model": 110, "brand_only": 150, "neither": 188}   # None = take all
BUCKET_W = pd.Series({"1-2": 1.0, "3-4": 1.0, "5-6": 1.5, "7+": 2.0})

def draw(df, n):
    if n is None or n >= len(df):
        return list(df.index)
    cnt = df["bucket"].value_counts().reindex(BUCKET_W.index).fillna(0)
    w = cnt * BUCKET_W
    alloc = np.minimum((w / w.sum() * n).round().astype(int), cnt.astype(int))
    chosen = []
    for b, k in alloc.items():
        ids = df.index[df["bucket"] == b].to_numpy()
        chosen += list(rng.choice(ids, size=int(k), replace=False))
    if len(chosen) < n:                                    # top up from whatever is left
        rest = df.index.difference(chosen).to_numpy()
        chosen += list(rng.choice(rest, size=min(n - len(chosen), len(rest)), replace=False))
    return chosen[:n]

picked = []
for st, n in QUOTA.items():
    picked += draw(pool[pool["stratum6"] == st], n)
samp = pool.loc[picked].copy()

title_of = us_products.set_index("product_id")["product_title"]
rows = us_examples[us_examples["query_id"].isin(samp["query_id"]) & us_examples["esci_label"].isin(["E", "S"])].sort_values(["query_id", "esci_label"])
ex_titles = rows.groupby("query_id")["product_id"].apply(lambda s: " || ".join(title_of.reindex(s.iloc[:2]).fillna("").str.slice(0, 90)))
samp["example_titles"] = samp["query_id"].map(ex_titles).fillna("")

blind = samp.sample(frac=1, random_state=11).reset_index(drop=True)
blind["label_id"] = np.arange(1, len(blind) + 1)
blind[["label_id", "query", "example_titles"]].assign(intent="", price="", note="").to_csv("classifier_pool_to_label.csv", index=False)
blind[["label_id", "query_id", "stratum6", "bucket", "heuristic", "source", "scope_group"]].to_csv("classifier_pool_key.csv", index=False)
print(len(blind), "queries to label\n"); print(pd.crosstab(blind["stratum6"], blind["bucket"], margins=True))
print("\nwrote classifier_pool_to_label.csv and classifier_pool_key.csv")

554 queries to label

bucket      1-2  3-4  5-6  7+  All
stratum6                          
NL_all        0    0    0  74   74
brand_only   29   78   37   6  150
model        40   45   18   7  110
neither      47   91   39  11  188
price_all     4   14   13   1   32
All         120  228  107  99  554

wrote classifier_pool_to_label.csv and classifier_pool_key.csv


In [ ]:
# 6C. RUN AFTER LABELING: QA, label counts, and the 70 / 15 / 15 classifier split (stratified on `intent`)
# keep_default_na=False: otherwise pandas turns the label "N/A" (and "NA") into a missing value and silently drops those rows
lab = pd.read_csv("classifier_pool_labeled.csv", dtype=str, keep_default_na=False); lab["label_id"] = lab["label_id"].astype(int)
key = pd.read_csv("classifier_pool_key.csv")
d = lab.merge(key, on="label_id")
norm = {"exact": "Exact", "semantic": "Semantic", "hybrid": "Hybrid", "na": "N/A", "n/a": "N/A"}
d["intent"] = d["intent"].astype(str).str.strip().str.lower().map(norm)
d["price"] = pd.to_numeric(d["price"], errors="coerce").fillna(0).astype(int)
print("unlabeled / unreadable rows:", int(d["intent"].isna().sum()))
d = d.dropna(subset=["intent"])
print("\nintent counts:\n", d["intent"].value_counts())
print("\nN/A rate by stratum:\n", d.groupby("stratum6")["intent"].apply(lambda s: (s == "N/A").mean()).round(3))
print("\nintent x stratum:\n", pd.crosstab(d["stratum6"], d["intent"]))
print("\nintent x price flag:\n", pd.crosstab(d["intent"], d["price"]))
use = d[d["intent"].isin(["Exact", "Semantic", "Hybrid"])].copy()
print("\nusable labeled queries:", len(use), "(need >= 500)")
mn = use["intent"].value_counts()
if len(mn) < 3 or mn.min() < 60:
    print("WARNING: smallest class has", int(mn.min()) if len(mn) else 0, "examples -> label more from the weakest stratum before training")

def strat_split(df, col, fr=(0.70, 0.15, 0.15), seed=42):
    r = np.random.default_rng(seed); tr, va, te = [], [], []
    for _, g in df.groupby(col):
        idx = r.permutation(g.index.to_numpy()); n = len(g)
        a, b = int(round(fr[0] * n)), int(round((fr[0] + fr[1]) * n))
        tr += list(idx[:a]); va += list(idx[a:b]); te += list(idx[b:])
    return df.loc[tr], df.loc[va], df.loc[te]

tr, va, te = strat_split(use, "intent")
for name, part in [("train", tr), ("val", va), ("test", te)]:
    part[["query_id", "query", "intent", "price"]].to_csv(f"classifier_{name}.csv", index=False)
    print(f"{name}: {len(part)}", part["intent"].value_counts().to_dict())

unlabeled / unreadable rows: 0

intent counts:
 intent
Hybrid      224
Semantic    204
Exact       123
N/A           3
Name: count, dtype: int64

N/A rate by stratum:
 stratum6
NL_all        0.000
brand_only    0.000
model         0.009
neither       0.011
price_all     0.000
Name: intent, dtype: float64

intent x stratum:
 intent      Exact  Hybrid  N/A  Semantic
stratum6                                
NL_all          0      20    0        54
brand_only     37     112    0         1
model          70      25    1        14
neither        15      62    2       109
price_all       1       5    0        26

intent x price flag:
 price       0   1
intent           
Exact     121   2
Hybrid    218   6
N/A         3   0
Semantic  167  37

usable labeled queries: 551 (need >= 500)
train: 386 {'Hybrid': 157, 'Semantic': 143, 'Exact': 86}
val: 82 {'Hybrid': 33, 'Semantic': 30, 'Exact': 19}
test: 83 {'Hybrid': 34, 'Semantic': 31, 'Exact': 18}


## ⏸ Stop here
1. Run **6B** now (kernel from Step 5 still valid) and send me the crosstab it prints.
2. Label `classifier_pool_to_label.csv` (≈ 550 rows; fill `intent` and `price`), save as **`classifier_pool_labeled.csv`**, run **6C** and send me its printout.
3. Please tell your supervisor about the Hybrid-definition change (price → lexical specificity + separate price flag) and send me their reaction; if they prefer the original definition, `price` + `intent` let me derive it without relabeling.